# Bronze Layer


## Storage Configuration

### ADLS Gen2 Connection Setup

### Storage Account
Defines the Azure Data Lake Storage account used for data access.

### Bronze Layer Path
Specifies the base path for the Bronze container where raw data is stored.

In [0]:
storage_account = "finalecommercerohit"

bronze_base_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/"

## Logging Setup

### Logger Initialization

### Purpose
Initializes a logger to track Bronze ingestion process.

### Start Log
Indicates the beginning of the Bronze data ingestion.

In [0]:
import logging
from pyspark.sql.functions import *
from pyspark.sql.types import *


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze_ingestion")

logger.info("Bronze ingestion started")

INFO:bronze_ingestion:Bronze ingestion started


# Carts Data Ingestion

## Read JSON Data

### Description
Loads carts data from Bronze storage in JSON format.

### Error Handling
Logs error if data loading fails.

In [0]:
try:
    carts_df = spark.read \
        .format("json") \
        .option("multiLine", "true") \
        .load(f"{bronze_base_path}/carts/")

    logger.info("Carts data loaded successfully")

except Exception as e:
    logger.error(f"Error reading carts data: {str(e)}")
    raise e

INFO:py4j.clientserver:Received command c on object id p0
INFO:bronze_ingestion:Carts data loaded successfully


# Products Data Ingestion

## Read JSON Data

### Description
Loads products data from Bronze storage in JSON format.

### Error Handling
Logs error if data loading fails.

In [0]:
try:
    products_df = spark.read \
        .format("json") \
        .option("multiLine", "true") \
        .load(f"{bronze_base_path}/Products/")

    logger.info("Products data loaded successfully")

except Exception as e:
    logger.error(f"Error reading products data: {str(e)}")
    raise e

INFO:bronze_ingestion:Products data loaded successfully


In [0]:
display(products_df.limit(10))


## Schema Inspection

## View Data Structure

### Description
Displays the schema of the products dataset.

In [0]:
products_df.printSchema()

INFO:py4j.clientserver:Received command c on object id p0


root
 |-- limit: long (nullable = true)
 |-- products: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- availabilityStatus: string (nullable = true)
 |    |    |-- brand: string (nullable = true)
 |    |    |-- category: string (nullable = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- dimensions: struct (nullable = true)
 |    |    |    |-- depth: double (nullable = true)
 |    |    |    |-- height: double (nullable = true)
 |    |    |    |-- width: double (nullable = true)
 |    |    |-- discountPercentage: double (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- images: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- meta: struct (nullable = true)
 |    |    |    |-- barcode: string (nullable = true)
 |    |    |    |-- createdAt: string (nullable = true)
 |    |    |    |-- qrCode: string (nullable = true)
 |    |    |    |-- updatedAt: string 

In [0]:
display(carts_df.limit(10))

In [0]:
carts_df.printSchema()

INFO:py4j.clientserver:Received command c on object id p0


root
 |-- carts: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- discountedTotal: double (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- products: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- discountPercentage: double (nullable = true)
 |    |    |    |    |-- discountedTotal: double (nullable = true)
 |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- price: double (nullable = true)
 |    |    |    |    |-- quantity: long (nullable = true)
 |    |    |    |    |-- thumbnail: string (nullable = true)
 |    |    |    |    |-- title: string (nullable = true)
 |    |    |    |    |-- total: double (nullable = true)
 |    |    |-- total: double (nullable = true)
 |    |    |-- totalProducts: long (nullable = true)
 |    |    |-- totalQuantity: long (nullable = true)
 |    |    |-- userId: long (nullable = true)
 |-- limit: long (nullable 

## Write to Bronze Tables

## Save Data to Unity Catalog

### Description
Writes carts and products data into Bronze tables.

### Output
Tables created in the Bronze layer of Unity Catalog.

In [0]:
carts_df.write.mode("append").saveAsTable("ecom_catalog.bronze.carts_bronze")

products_df.write.mode("append").saveAsTable("ecom_catalog.bronze.products_bronze")

logger.info("Bronze tables created in Unity Catalog")

INFO:bronze_ingestion:Bronze tables created in Unity Catalog


## Write to ADLS

## Store Data in Bronze Layer

### Description
Saves carts and products data as Parquet files in ADLS Bronze storage.

### Output
Data stored in Bronze zone for further processing.

In [0]:
carts_df.write.mode("overwrite").parquet(f"{bronze_base_path}tables/carts_bronze")

products_df.write.mode("overwrite").parquet(f"{bronze_base_path}tables/products_bronze")

logger.info("Data also stored in ADLS Bronze zone")

INFO:bronze_ingestion:Data also stored in ADLS Bronze zone


# Incremental Load

In [0]:
# ============================
# STEP 1: BASELINE SNAPSHOT
# ============================

carts_count = spark.read.table("ecom_catalog.bronze.carts_bronze").count()
products_count = spark.read.table("ecom_catalog.bronze.products_bronze").count()

print("=== BASELINE (CURRENT STATE) ===")
print("CARTS ROW COUNT:", carts_count)
print("PRODUCTS ROW COUNT:", products_count)

=== BASELINE (CURRENT STATE) ===
CARTS ROW COUNT: 13
PRODUCTS ROW COUNT: 13


## Incremental Load Validation

## Read New Data

### Description
Loads new carts and products data from Bronze storage.

## Row Count Simulation

### Description
Estimates total records after adding new data.

## Incremental Proof

### Description
Displays count of newly added records.

In [0]:

from pyspark.sql.functions import *

storage_account = "finalecommercerohit"
bronze_base_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/"


carts_new = spark.read.option("multiLine", "true") \
    .json(f"{bronze_base_path}carts/")

products_new = spark.read.option("multiLine", "true") \
    .json(f"{bronze_base_path}Products/")



carts_after = carts_before + carts_new.count()
products_after = products_before + products_new.count()

print("\n=== AFTER ADDING NEW FILES (SIMULATION) ===")
print("CARTS COUNT:", carts_after)
print("PRODUCTS COUNT:", products_after)



print("\n=== INCREMENTAL LOAD PROOF ===")
print("New carts added:", carts_new.count())
print("New products added:", products_new.count())


=== AFTER ADDING NEW FILES (SIMULATION) ===
CARTS COUNT: 15
PRODUCTS COUNT: 16

=== INCREMENTAL LOAD PROOF ===
New carts added: 2
New products added: 3
